# Exploration: PyTerrier prebuilt BM25 indexes for the GRPO query-expansion project
Question: are `pyterrier/{nfcorpus,scifact,fiqa}.terrier` usable as the frozen BM25 retriever inside the GRPO reward loop?
Checks: (1) load, (2) title field, (3) qrels/splits, (4) accuracy vs published BM25, (5) speed (gate: 64 queries < 1 s), (6) robustness to expansion-style text, (7) verdict.

## 1. Setup

In [1]:
import os, glob, subprocess, time, re, warnings
warnings.filterwarnings("ignore")

# Java is installed but not on PATH on this machine; point JAVA_HOME at a JDK >= 11.
if "JAVA_HOME" not in os.environ:
    for c in ["/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home",
              "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",
              "/Library/Java/JavaVirtualMachines/amazon-corretto-11.jdk/Contents/Home"]:
        if os.path.exists(c + "/bin/java"):
            os.environ["JAVA_HOME"] = c
            break
print("JAVA_HOME =", os.environ.get("JAVA_HOME"))
print(subprocess.run([os.environ["JAVA_HOME"] + "/bin/java", "-version"], capture_output=True, text=True).stderr.strip().splitlines()[-3:])

import numpy as np, pandas as pd, ir_datasets, pyterrier as pt
from ranx import Qrels, Run, evaluate
print("pyterrier", pt.__version__)

JAVA_HOME = /opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home
['openjdk version "21.0.12.1" 2026-08-18', 'OpenJDK Runtime Environment Homebrew (build 21.0.12.1)', 'OpenJDK 64-Bit Server VM Homebrew (build 21.0.12.1, mixed mode, sharing)']


pyterrier 1.1.2


## 2. Load the prebuilt indexes

In [2]:
DATASETS = ["nfcorpus", "scifact", "fiqa"]
EXPECTED_DOCS = {"nfcorpus": 3633, "scifact": 5183, "fiqa": 57638}
indexes, stats = {}, []
for ds in DATASETS:
    t0 = time.time()
    idx = pt.Artifact.from_hf(f"pyterrier/{ds}.terrier")
    obj = idx.index_obj() if hasattr(idx, "index_obj") else idx
    cs = obj.getCollectionStatistics()
    indexes[ds] = idx
    stats.append(dict(dataset=ds, docs=int(cs.getNumberOfDocuments()), expected=EXPECTED_DOCS[ds],
                      tokens=int(cs.getNumberOfTokens()), avg_len=round(float(cs.getAverageDocumentLength()), 1),
                      meta_keys=list(obj.getMetaIndex().getKeys()), load_s=round(time.time() - t0, 1)))
pd.DataFrame(stats)

Java started (triggered by TerrierIndex.index_ref) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


,dataset,docs,expected,tokens,avg_len,meta_keys,load_s
0,nfcorpus,3633,3633,567901,156.3,"[docno, text]",0.4
1,scifact,5183,5183,745811,143.9,"[docno, text]",0.0
2,fiqa,57638,57638,3783214,65.6,"[docno, text]",0.0


## 3. Does the stored text include the title?
Compare the stored `text` meta field with BEIR `title` and `text` from ir_datasets.

In [3]:
def meta_text(idx, docno):
    obj = idx.index_obj()
    docid = obj.getMetaIndex().getDocument("docno", docno)
    return obj.getMetaIndex().getItem("text", docid)

rows = []
for ds in DATASETS:
    d = ir_datasets.load(f"beir/{ds}")
    docs = {}
    for i, doc in enumerate(d.docs_iter()):
        docs[doc.doc_id] = doc
        if i >= 200: break
    ids = list(docs)[:20]
    n_title = n_exact_text = 0
    for did in ids:
        stored = meta_text(indexes[ds], did)
        doc = docs[did]
        title = getattr(doc, "title", "") or ""
        if title and title.strip()[:40] in stored: n_title += 1
        if stored.strip() == doc.text.strip(): n_exact_text += 1
    rows.append(dict(dataset=ds, checked=len(ids), stored_contains_title=n_title, stored_equals_body_only=n_exact_text,
                     doc_fields=d.docs_cls()._fields))
pd.DataFrame(rows)

[INFO] Opening /Users/riddhizantye/.ir_datasets/beir/nfcorpus/docs.pklz4/bin with direct file access


[INFO] Opening /Users/riddhizantye/.ir_datasets/beir/scifact/docs.pklz4/bin with direct file access
[INFO] Opening /Users/riddhizantye/.ir_datasets/beir/fiqa/docs.pklz4/bin with direct file access


,dataset,checked,stored_contains_title,stored_equals_body_only,doc_fields
0,nfcorpus,20,20,0,"(doc_id, text, title, url)"
1,scifact,20,20,0,"(doc_id, text, title)"
2,fiqa,20,0,20,"(doc_id, text)"


In [4]:
ds = "nfcorpus"; d = ir_datasets.load(f"beir/{ds}")
doc = next(iter(d.docs_iter()))
print("TITLE:", doc.title); print("BODY :", doc.text[:150]); print("STORED:", meta_text(indexes[ds], doc.doc_id)[:250])

TITLE: Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland
BODY : Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast can
STORED: Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the


[INFO] Opening /Users/riddhizantye/.ir_datasets/beir/nfcorpus/docs.pklz4/bin with direct file access


## 4. Queries, qrels and split hygiene

In [5]:
SPLITS = {"nfcorpus": ["train", "dev", "test"], "scifact": ["train", "test"], "fiqa": ["train", "dev", "test"]}
data = {}   # data[ds][split] = (queries_df, qrels_dict)
for ds in DATASETS:
    data[ds] = {}
    for sp in SPLITS[ds]:
        try:
            d = ir_datasets.load(f"beir/{ds}/{sp}")
            q = pd.DataFrame([(x.query_id, x.text) for x in d.queries_iter()], columns=["qid", "query"])
            qr = {}
            for r in d.qrels_iter():
                if r.relevance > 0: qr.setdefault(r.query_id, {})[r.doc_id] = int(r.relevance)
            data[ds][sp] = (q, qr)
        except Exception as e:
            print(f"{ds}/{sp}: unavailable ({type(e).__name__}: {e})")

rows = []
for ds in DATASETS:
    all_ids = [set(v[0].qid) for v in data[ds].values()]
    overlap = sum(len(a & b) for i, a in enumerate(all_ids) for b in all_ids[i+1:])
    obj = indexes[ds].index_obj(); meta = obj.getMetaIndex()
    for sp, (q, qr) in data[ds].items():
        missing = sum(1 for rels in qr.values() for did in rels if meta.getDocument("docno", did) < 0)
        rows.append(dict(dataset=ds, split=sp, queries=len(q), with_qrels=len(qr), qrel_docs_missing_from_index=missing, split_overlap=overlap))
pd.DataFrame(rows)

,dataset,split,queries,with_qrels,qrel_docs_missing_from_index,split_overlap
0,nfcorpus,train,2590,2590,0,0
1,nfcorpus,dev,324,324,0,0
2,nfcorpus,test,323,323,0,0
3,scifact,train,809,809,0,0
4,scifact,test,300,300,0,0
5,fiqa,train,5500,5500,0,0
6,fiqa,dev,500,500,0,0
7,fiqa,test,648,648,0,0


## 5. Accuracy of `index.bm25()` vs published numbers
Model card: NFCorpus dev 0.2944, test 0.3282. Published BEIR BM25 nDCG@10: ~0.325 / 0.665 / 0.236.
R_cap@100 = relevant docs in top 100 / min(100, #relevant).

In [6]:
def rcap100(run_df, qrels):
    out = {}
    for qid, g in run_df.groupby("qid"):
        rel = qrels.get(qid, {})
        if not rel: continue
        hits = sum(1 for d_ in g.sort_values("rank").docno.head(100) if d_ in rel)
        out[qid] = hits / min(100, len(rel))
    return float(np.mean(list(out.values())))

def clean(q):
    return re.sub(r"\s+", " ", re.sub(r"[^A-Za-z0-9 ]", " ", q)).strip()

def run_bm25(ds, queries_df, k=100, sanitize=True):
    bm25 = indexes[ds].bm25(num_results=k)
    qdf = queries_df.copy()
    if sanitize: qdf["query"] = qdf["query"].map(clean)
    return bm25.transform(qdf)

def evaluate_split(ds, sp):
    q, qr = data[ds][sp]
    q = q[q.qid.isin(qr)]
    run = run_bm25(ds, q)
    run = run.astype({"qid": object, "docno": object, "score": "float64"})
    r = Run.from_df(run, q_id_col="qid", doc_id_col="docno", score_col="score")
    m = evaluate(Qrels.from_dict(qr), r, ["ndcg@10", "recall@100", "mrr@10"], make_comparable=True)
    return dict(dataset=ds, split=sp, n=len(q), queries_with_no_results=int(len(q) - run.qid.nunique()), ndcg10=round(m["ndcg@10"], 4), recall100=round(m["recall@100"], 4),
                rcap100=round(rcap100(run, qr), 4), mrr10=round(m["mrr@10"], 4))

PUBLISHED = {"nfcorpus": 0.325, "scifact": 0.665, "fiqa": 0.236}
acc = pd.DataFrame([evaluate_split(ds, sp) for ds in DATASETS for sp in data[ds] if sp != "train"])
acc["published_test_ndcg10"] = acc.dataset.map(PUBLISHED)
acc

,dataset,split,n,queries_with_no_results,ndcg10,recall100,rcap100,mrr10,published_test_ndcg10
0,nfcorpus,dev,324,21,0.2943,0.2249,0.2495,0.4938,0.325
1,nfcorpus,test,323,15,0.3291,0.2483,0.2745,0.5321,0.325
2,scifact,test,300,0,0.6839,0.9260,0.9260,0.6441,0.665
3,fiqa,dev,500,0,0.2695,0.5475,0.5475,0.3289,0.236
4,fiqa,test,648,0,0.2526,0.5592,0.5592,0.3103,0.236


## 6. Speed
Gate from the plan: 64 queries (top-100) in under 1 second. Uses raw train queries, then expansion-length queries.

In [7]:
q_train, _ = data["nfcorpus"]["train"]
q64 = q_train.head(64).copy(); q64["query"] = q64["query"].map(clean)
bm25 = indexes["nfcorpus"].bm25(num_results=100)
bm25.transform(q64.head(2))  # warm-up
times = []
for _ in range(5):
    t0 = time.time(); bm25.transform(q64); times.append(time.time() - t0)
print("64 short queries, s:", [round(t, 3) for t in times])

# Expansion-like queries: original + ~40 extra words (reuse words from other queries)
rng = np.random.default_rng(0)
vocab = " ".join(q_train["query"].map(clean)).split()
q64x = q64.copy()
q64x["query"] = q64x["query"].map(lambda s: s + " " + " ".join(rng.choice(vocab, 40)))
times_x = []
for _ in range(5):
    t0 = time.time(); bm25.transform(q64x); times_x.append(time.time() - t0)
print("64 expanded queries (+40 words), s:", [round(t, 3) for t in times_x])

64 short queries, s: [0.11, 0.108, 0.107, 0.11, 0.108]


64 expanded queries (+40 words), s: [0.158, 0.156, 0.156, 0.156, 0.158]


## 7. Robustness to LLM-style text
Raw Terrier query parsing treats `:`, `"`, `(`, `)`, `^`, `~`, `/` etc. as syntax. Try unsanitized queries.

In [8]:
tricky = [
    "breast cancer: risk factors (diet) \"fiber\"",
    "what's the effect of vitamin D/calcium on bone-density?",
    "cancer AND (diet OR exercise) NOT smoking",
    "50% reduction in LDL; statins^2 ~ 3",
    "alpha-synuclein [PD] {Parkinson's} <query>tags</query>",
    "x " * 400,
    "",
]
rows = []
for s in tricky:
    for san in (False, True):
        try:
            df = pd.DataFrame([{"qid": "t1", "query": clean(s) if san else s}])
            res = indexes["nfcorpus"].bm25(num_results=100).transform(df)
            rows.append(dict(query=s[:45], sanitized=san, status="ok", n_results=len(res)))
        except Exception as e:
            rows.append(dict(query=s[:45], sanitized=san, status=f"{type(e).__name__}: {str(e)[:70]}", n_results=None))
pd.DataFrame(rows)

,query,sanitized,status,n_results
0,"breast cancer: risk factors (diet) ""fiber""",False,ok,100
1,"breast cancer: risk factors (diet) ""fiber""",True,ok,100
2,what's the effect of vitamin D/calcium on bon,False,ok,100
3,what's the effect of vitamin D/calcium on bon,True,ok,100
4,cancer AND (diet OR exercise) NOT smoking,False,ok,100
5,cancer AND (diet OR exercise) NOT smoking,True,ok,100
6,50% reduction in LDL; statins^2 ~ 3,False,ok,100
7,50% reduction in LDL; statins^2 ~ 3,True,ok,100
8,alpha-synuclein [PD] {Parkinson's} <query>tag,False,ok,100
9,alpha-synuclein [PD] {Parkinson's} <query>tag,True,ok,100


## 8. Verdict

In [9]:
ok_speed = float(np.median(times_x)) < 1.0
a = acc[acc.split == "test"].set_index("dataset")
print("Median time, 64 expanded queries: %.3fs  (gate < 1s: %s)" % (np.median(times_x), "PASS" if ok_speed else "FAIL"))
for ds in DATASETS:
    print(f"{ds:9s} test nDCG@10 = {a.loc[ds,'ndcg10']:.4f}  (published ~{PUBLISHED[ds]})")
print("Index doc counts match expected:", all(s["docs"] == s["expected"] for s in stats))

Median time, 64 expanded queries: 0.156s  (gate < 1s: PASS)
nfcorpus  test nDCG@10 = 0.3291  (published ~0.325)
scifact   test nDCG@10 = 0.6839  (published ~0.665)
fiqa      test nDCG@10 = 0.2526  (published ~0.236)
Index doc counts match expected: True


**Verdict (written after reading the outputs above)**

PyTerrier's prebuilt indexes are viable as the frozen BM25 retriever.
- **Counts and joins:** doc counts match BEIR exactly (3,633 / 5,183 / 57,638). Every qrel doc is in the index, and splits are disjoint.
- **Fields:** NFCorpus and SciFact stored text includes the title. FiQA has no titles in BEIR, so the index is body only, which is the correct match.
- **Accuracy:** dev and test nDCG@10 are 0.2943 / 0.3291 on NFCorpus (model card: 0.2944 / 0.3282), 0.6839 on SciFact test and 0.2526 on FiQA test. All are within about 0.02 of published BEIR BM25 and slightly above.
- **Speed:** 64 queries at top-100 take about 0.11 s (short) and 0.16 s (+40 words), against the 1 s gate.
- **Parser:** no punctuation, boolean or markup input crashed the parser, even unsanitized. I still sanitize, since it is cheap and makes the scored string explicit.
- **Zero-result queries:** 21 NFCorpus dev and 15 test queries get no results from raw BM25. They are single out-of-vocabulary words (Zoloft, okra, Yale, ...), so they score 0 on every metric. Expansion is the only way these can score, so treat them as the headline case in the error analysis and report them separately.

Recommendation: use PyTerrier and drop bm25s. Open items: Java must be on the training box (JarvisLabs images may need a JDK install), and the Terrier analyzer (stemming and stopwords) is whatever the prebuilt index used, so it cannot be changed without rebuilding.